# DecisionFormer quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FrontiersMindAI/decisionformer/blob/main/notebooks/quickstart.ipynb)

This notebook walks through the whole loop, from a pretrained model to your own fine-tune:

1. **Try a model**: ask `vllm-sr/d3-edge` typed questions and get a probability for every option.
2. **Ask your own questions**: edit a request and batch several with `decide_many`.
3. **Prepare data**: split `FrontiersMind/deciformer-15k` into train, calibration and dev sets.
4. **Fine-tune**: train a small model from a bundled recipe.
5. **Evaluate**: measure accuracy and calibration on the dev set and plot the training curves.
6. **Use your model**: load the checkpoint and ask the same questions as in step 1.
7. **Ship it** (optional): push to the Hub or serve it on `/v1/systemone`.

Each section runs on its own once **Setup** has run, so you can skip what you don't need. Everything you might
want to change is in the **Settings** cell. With the defaults, training takes about 20-30 minutes on a free Colab T4
GPU and about an hour on a laptop CPU; lower `MAX_STEPS` or `TRAIN_LIMIT` for a quicker (and weaker) run.

## 0. Setup

In [ ]:
# Inside a clone of the repo this installs it in editable mode; anywhere else (Colab) it installs from GitHub.
import os
from pathlib import Path

REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "decisionformer").is_dir()), None)
if REPO_ROOT is not None:
    %pip install -q -e "{REPO_ROOT}[data,serve]"
else:
    %pip install -q "decisionformer[data,serve] @ git+https://github.com/FrontiersMindAI/decisionformer.git"
%pip install -q matplotlib

### Settings

`RECIPE` picks the training setup (see `decisionformer recipes` for the full list). By default it follows the
hardware:

| recipe | model | mode | default on | steps | MASSIVE (18 options) after training |
|---|---|---|---|---|---|
| `deciformer15k_qwen3` | Qwen3-0.6B | LoRA | a GPU | 375 (one epoch) | 0.86 en-US, 0.82 de-DE (`docs/benchmarks.md`) |
| `local_smollm2` | SmolLM2-135M-Instruct | full fine-tune | CPU or Apple silicon | 300 | much lower: a small model needs many steps |

**How much to train.** The model sees `steps × batch size` examples (batch size 32 for the Qwen recipe, 16 for
SmolLM2). `MAX_STEPS = None` keeps the recipe's own number. Short runs are fine for checking that everything works,
but questions with many options (MASSIVE has 18) need the full run: a 100-step SmolLM2 run gave every MASSIVE row
the same answer and scored 0%.

**Which data.** deciformer-15k has 9 tasks: `aegis2`, `boolq`, `civil_comments`, `massive-de-DE`, `massive-en-US`,
`multinli`, `paws`, `squad2` and `vitaminc-dev`.

- `TRAIN_TASKS` and `EVAL_TASKS` keep only the listed tasks (`None` keeps all of them). The calibration set follows
  `TRAIN_TASKS`.
- `TRAIN_LIMIT` and `EVAL_LIMIT` keep a random sample of that many records (`None` keeps all of them).
- The evaluation settings apply both to the evaluations during training and to section 5.

`TRAIN_OVERRIDES` takes any other dotted config key, exactly like the command line (`train.lr=1e-4`).

In [ ]:
import torch

PRETRAINED_MODEL = "vllm-sr/d3-edge"     # used in sections 1 and 2
RECIPE = "deciformer15k_qwen3" if torch.cuda.is_available() else "local_smollm2"
MAX_STEPS = None                         # None: the recipe's own (375 for Qwen3, 300 for SmolLM2)

TRAIN_TASKS = None                       # e.g. ["massive-en-US", "massive-de-DE"]; None trains on all 9 tasks
TRAIN_LIMIT = None                       # e.g. 2000 training records; None uses all 11,982
EVAL_TASKS = None                        # e.g. ["massive-en-US"]; None evaluates all tasks
EVAL_LIMIT = 300                         # dev records to evaluate; None uses all 1,509

TRAIN_OVERRIDES = {}                     # e.g. {"train.lr": 1e-4, "train.eval_every": 50}
DATA_DIR = "data/deciformer-15k"
OUTPUT_DIR = "runs/notebook-quickstart"
HUB_REPO = None                          # e.g. "my-org/my-decider" to push in section 7

# Data and runs land next to the repo (both are gitignored) or in the current folder outside a clone.
WORKDIR = REPO_ROOT or Path.cwd()
os.chdir(WORKDIR)
print("working directory:", WORKDIR)

In [ ]:
import json
import logging
import random
import subprocess
import sys
from collections import Counter

import decisionformer
from decisionformer import AutoDecisionModel

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s", datefmt="%H:%M:%S", force=True)

if torch.cuda.is_available():
    device = f"cuda ({torch.cuda.get_device_name(0)}, bf16: {torch.cuda.is_bf16_supported()})"
elif torch.backends.mps.is_available():
    device = "mps (Apple silicon, float32)"
else:
    device = "cpu (float32)"
print("decisionformer", decisionformer.__version__, "| torch", torch.__version__, "| device:", device)


def cli(*args, quiet=False):
    # Runs `decisionformer ARGS...` and returns its standard output (printed unless quiet).
    result = subprocess.run([sys.executable, "-m", "decisionformer", *map(str, args)], capture_output=True,
                            text=True, encoding="utf-8")
    if not quiet:
        print(result.stdout, end="")
    if result.returncode:
        raise RuntimeError(f"decisionformer {args[0]} failed:\n{result.stderr[-3000:]}")
    return result.stdout


def subset(source, target, tasks=None, limit=None, seed=0):
    # Writes the records of `source` whose task is in `tasks` to `target`, a random sample of `limit` of them
    # if given, keeping their original order. Returns the path and the number of records per task.
    with open(source, encoding="utf-8") as stream:
        lines = [line for line in stream if not tasks or json.loads(line).get("task") in tasks]
    if limit and limit < len(lines):
        keep = set(random.Random(seed).sample(range(len(lines)), limit))
        lines = [line for number, line in enumerate(lines) if number in keep]
    if not lines:
        raise ValueError(f"no records in {source} for tasks {tasks}")
    Path(target).write_text("".join(lines), encoding="utf-8")
    return str(target), Counter(json.loads(line).get("task") for line in lines)

The device is picked automatically: a CUDA GPU, then Apple silicon, then the CPU. Set the environment variable
`DECISIONFORMER_DEVICE=cpu` before importing to force the CPU. GPUs without bf16 (such as Colab's T4) train in fp16
with loss scaling and log a warning about it; that is expected.

## 1. Try a pretrained decision model

A decision model takes a **state** (the text to judge) and typed **questions**, and returns a probability for
every option in one forward pass, without generating text. There are three question types:

| type | answers | `criteria` |
|---|---|---|
| `choice` | one of several options | a dict of option name to description |
| `noul` | yes or no, as the probability of yes | none |
| `score` | one of 2 to 10 ordered levels | a list of level descriptions, lowest first |

`vllm-sr/d3-edge` has 0.6B parameters: expect a few seconds per request on a CPU.

In [ ]:
model = AutoDecisionModel.from_pretrained(PRETRAINED_MODEL)

STATE = "The order arrived damaged yesterday. The customer has a receipt and asks for a replacement today."
QUESTIONS = {
    "route": {
        "type": "choice",
        "instructions": "Which team should handle this request?",
        "criteria": {
            "returns": "Refunds, replacements and damaged deliveries",
            "billing": "Payments, invoices and charges",
            "technical": "Product setup and faults",
        },
    },
    "receipt": {
        "type": "noul",
        "instructions": "Does the customer have a receipt?",
    },
    "urgency": {
        "type": "score",
        "instructions": "How urgent is this request?",
        "criteria": ["Routine", "Soon", "Today"],
    },
}

response = model.decide(state=STATE, questions=QUESTIONS)
print(json.dumps(response, indent=2))

Reading the response:

- `choice` is the top option and `probabilities` covers every option.
- `noul` is the probability that the answer is yes.
- `score` is the expected level scaled to 0..1, and `legend` maps level numbers back to your descriptions.
- `confidence` is **not** the top probability. It is 1 minus the normalized entropy: 0 when all options are
  equally likely, 1 when the model is certain. A 0.51/0.49 split has a confidence near 0.

## 2. Ask your own questions

Edit the request below. `decide_many` answers several requests at once, and their questions share forward
passes. Requests longer than the model's context are refused with a `ContextLengthError` instead of being
truncated; pass `max_length=` to `from_pretrained` to allow longer ones.

In [ ]:
requests = [
    {
        "state": "I was charged twice for my subscription this month. This is the third time!",
        "questions": {
            "angry": {"type": "noul", "instructions": "Is the customer angry?"},
            "topic": {"type": "choice", "criteria": {"billing": "Payments and invoices",
                                                     "shipping": "Deliveries and tracking",
                                                     "account": "Login and profile"}},
        },
    },
    {
        "state": "Where can I change the email address on my account?",
        "questions": {
            "angry": {"type": "noul", "instructions": "Is the customer angry?"},
            "topic": {"type": "choice", "criteria": {"billing": "Payments and invoices",
                                                     "shipping": "Deliveries and tracking",
                                                     "account": "Login and profile"}},
        },
    },
]

for request, answer in zip(requests, model.decide_many(requests)):
    answers = answer["answers"]
    print(request["state"])
    print(f"  angry: {answers['angry']['noul']:.3f}   topic: {answers['topic']['choice']} "
          f"(confidence {answers['topic']['confidence']:.2f})\n")

In [ ]:
# Free the memory before training.
del model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 3. Prepare training data

Training data is one JSON object per line: the request plus the gold answers. A minimal record:

```json
{"id": "t-17", "family": "ticket-17",
 "state": "I was charged twice this month.",
 "questions": {"route": {"type": "choice", "criteria": {"billing": "Payments and invoices", "returns": null}},
               "angry": {"type": "noul", "instructions": "Is the customer angry?"}},
 "gold": {"route": "billing", "angry": false}}
```

`family` keeps related records together when splitting. Optional `targets` add soft labels per question. Files
can be `.jsonl`, `.jsonl.gz` or a Hub dataset written as `hf:<name>[:<config>]:<split>`.

The cell below splits `FrontiersMind/deciformer-15k` (9 decision tasks, English and German), holding out 10% for
calibration and 10% for dev and keeping whole families together. To use your own data, point it at your file.

In [ ]:
if not Path(DATA_DIR, "train.jsonl").exists():
    cli("data", "split", "hf:FrontiersMind/deciformer-15k:train", "--out", DATA_DIR, quiet=True)
else:
    print("already split:", DATA_DIR)

for name in ("train", "calibration", "dev"):
    with open(Path(DATA_DIR, f"{name}.jsonl"), encoding="utf-8") as stream:
        print(f"{name:12s}{sum(1 for _ in stream):>7,d} records")

Next, the files training and evaluation actually use are cut from the split according to `TRAIN_TASKS`,
`TRAIN_LIMIT`, `EVAL_TASKS` and `EVAL_LIMIT` in Settings. Rerun this cell after changing them.

In [ ]:
selection = Path(DATA_DIR, "selected")
selection.mkdir(exist_ok=True)
TRAIN_FILE, train_tasks = subset(f"{DATA_DIR}/train.jsonl", selection / "train.jsonl", TRAIN_TASKS, TRAIN_LIMIT)
CALIBRATION_FILE, _ = subset(f"{DATA_DIR}/calibration.jsonl", selection / "calibration.jsonl", TRAIN_TASKS)
EVAL_FILE, eval_tasks = subset(f"{DATA_DIR}/dev.jsonl", selection / "dev.jsonl", EVAL_TASKS, EVAL_LIMIT)

print(f"{'task':16s}{'train':>8s}{'eval':>8s}")
for task in sorted(set(train_tasks) | set(eval_tasks)):
    print(f"{task:16s}{train_tasks[task]:>8,d}{eval_tasks[task]:>8,d}")
print(f"{'total':16s}{sum(train_tasks.values()):>8,d}{sum(eval_tasks.values()):>8,d}")
untrained = set(eval_tasks) - set(train_tasks)
if untrained:
    print("\nnote: evaluating tasks the model is not trained on:", ", ".join(sorted(untrained)))

In [ ]:
# One training record, and a check that no ids, families or states leak between the splits.
with open(Path(DATA_DIR, "train.jsonl"), encoding="utf-8") as stream:
    print(json.dumps(json.loads(next(stream)), indent=2, ensure_ascii=False)[:1500])

check = json.loads(cli("data", "check", f"dev={EVAL_FILE}", f"calibration={CALIBRATION_FILE}",
                       f"train={TRAIN_FILE}", quiet=True))
print("\nleakage check:", "ok" if check["ok"] else "PROBLEMS")
for problem in check["problems"]:
    print(" -", problem)

## 4. Fine-tune

`decisionformer.train` takes a recipe name, a YAML file or a dict, plus dotted overrides. It evaluates on the dev
set during training, fits calibration temperatures on the calibration set, and writes:

- `best/` and `last/`: finished checkpoints in the standard Hugging Face layout
- `metrics.jsonl`: training loss and evaluation metrics
- `summary.json`: the best step and its metrics

Rerunning this cell resumes an unfinished run from its last saved state. To see the full resolved config first,
run `cli("config", RECIPE)`.

Expect about 20-30 minutes for the Qwen3 recipe on a T4, and about an hour for the SmolLM2 recipe on a 6-core CPU
(roughly 15 s per step). If you change the settings, use a new `OUTPUT_DIR`: rerunning with the same one resumes
or reuses the old run.

In [ ]:
overrides = {
    "data.train": TRAIN_FILE,
    "data.calibration": CALIBRATION_FILE,
    "data.eval": {"dev": EVAL_FILE},
    "train.output_dir": OUTPUT_DIR,
    **({"train.max_steps": MAX_STEPS} if MAX_STEPS else {}),
    **TRAIN_OVERRIDES,
}
print(f"training {RECIPE} on {sum(train_tasks.values()):,d} records, output in {OUTPUT_DIR}")
summary = decisionformer.train(RECIPE, overrides)
print(json.dumps(summary, indent=2))

## 5. Evaluate

In [ ]:
# Training curves from metrics.jsonl.
import matplotlib.pyplot as plt

with open(Path(OUTPUT_DIR, "metrics.jsonl"), encoding="utf-8") as stream:
    records = [json.loads(line) for line in stream]
train_rows = [r for r in records if r.get("event") == "train"]
eval_rows = [r for r in records if r.get("event") == "eval"]

fig, (loss_ax, acc_ax) = plt.subplots(1, 2, figsize=(11, 3.5))
loss_ax.plot([r["step"] for r in train_rows], [r["loss"] for r in train_rows])
loss_ax.set(title="training loss", xlabel="step")
for flavor in ("raw", "calibrated"):
    points = [(r["step"], r["sets"]["dev"][flavor]["overall"]["accuracy"])
              for r in eval_rows if flavor in r["sets"].get("dev", {})]
    if points:
        acc_ax.plot(*zip(*points), marker="o", label=flavor)
acc_ax.set(title="dev accuracy", xlabel="step")
acc_ax.legend()
plt.tight_layout()
plt.show()

`decisionformer eval` scores a checkpoint on labeled data and reports accuracy, expected calibration error
(`ece`), Brier score and negative log-likelihood, overall and per task and question type. `--predictions` saves
one row per question, which `decisionformer compare` uses for a paired bootstrap between two models.

The table puts two references next to each task's accuracy:

- **chance**: the accuracy of guessing uniformly (1/2 for yes/no, 1/18 for MASSIVE's intents).
- **top answer**: how often the model gave its most common answer for that task. Close to 100% on a task with
  several balanced options means the model has *collapsed*: it gives the same answer whatever the input, usually
  because it was trained too briefly. Train for longer or with a larger model.

In [ ]:
CHECKPOINT = f"{OUTPUT_DIR}/best"
PREDICTIONS = f"{OUTPUT_DIR}/dev.predictions.jsonl"
report = json.loads(cli("eval", CHECKPOINT, EVAL_FILE, "--predictions", PREDICTIONS, quiet=True))
overall = report["overall"]
print(f"overall: accuracy {overall['accuracy']:.4f}  ece {overall['ece']:.4f}  "
      f"brier {overall['brier']:.4f}  nll {overall['nll']:.4f}  ({overall['count']} questions)\n")

with open(PREDICTIONS, encoding="utf-8") as stream:
    by_task = {}
    for row in map(json.loads, stream):
        by_task.setdefault(row["task"], []).append(row)

print(f"{'task':16s}{'rows':>6s}{'accuracy':>10s}{'chance':>8s}{'top answer':>12s}")
for task, metrics in report["tasks"].items():
    rows = by_task.get(task, [])
    chance = sum(1 / len(r["probabilities"]) for r in rows) / max(len(rows), 1)
    answers = Counter(max(range(len(r["probabilities"])), key=r["probabilities"].__getitem__) for r in rows)
    top_share = answers.most_common(1)[0][1] / len(rows) if rows else 0.0
    flag = "  <- collapsed" if top_share > 0.9 and len(rows[0]["probabilities"]) > 2 else ""
    print(f"{task:16s}{len(rows):>6d}{metrics['accuracy']:>10.3f}{chance:>8.3f}{top_share:>11.0%}{flag}")

Optional: compare against the pretrained model from section 1 on the same rows. The two models use different
prompts, so this measures them as complete systems. `d3-edge` is slow on a CPU, so this cell only runs on a GPU
unless you change the condition.

In [ ]:
if torch.cuda.is_available():
    cli("eval", PRETRAINED_MODEL, EVAL_FILE, "--predictions", f"{OUTPUT_DIR}/dev.pretrained.predictions.jsonl",
        quiet=True)
    cli("compare", PREDICTIONS, f"{OUTPUT_DIR}/dev.pretrained.predictions.jsonl")
else:
    print("skipped on CPU")

## 6. Use your fine-tuned model

The checkpoint loads exactly like a Hub model. Here it answers the request from section 1.

In [ ]:
mine = AutoDecisionModel.from_pretrained(CHECKPOINT)
print(json.dumps(mine.decide(state=STATE, questions=QUESTIONS)["answers"], indent=2))

**If every option gets about the same probability** (confidence near 0), the model has not learned that question
type yet. Look at the run's calibration temperatures (printed below): calibration divides the scores by these, and a `choice` temperature at
its cap of 20 means the model's choice scores carried almost no information, so calibration flattened them. This
is what a short SmolLM2 run produces. Train for the recipe's full number of steps, or use the Qwen3 recipe on a GPU.

In [ ]:
temperatures = json.loads(Path(OUTPUT_DIR, "summary.json").read_text(encoding="utf-8"))["temperatures"]
print("calibration temperatures:", {k: round(v, 2) for k, v in temperatures.items()})

## 7. Ship it (optional)

**Push to the Hugging Face Hub.** Set `HUB_REPO` in Settings and log in first (`huggingface-cli login`, or
`from huggingface_hub import login; login()`). Repos are created private.

In [ ]:
if HUB_REPO:
    print(mine.push_to_hub(HUB_REPO))   # private=False to make it public
else:
    print("set HUB_REPO in the Settings cell to push")

**Serve it as an API.** The server speaks the System One API used by Jev, vLLM and the Decision Index, and its
schemas are at `/docs`. From a terminal:

```bash
decisionformer serve runs/notebook-quickstart/best --port 8000
```

The cell below starts the server in the background, sends one request and stops it.

In [ ]:
import time
import urllib.request

server = subprocess.Popen([sys.executable, "-m", "decisionformer", "serve", CHECKPOINT, "--port", "8000"])
try:
    body = json.dumps({"state": "I was charged twice for my subscription this month.",
                       "questions": {"angry": {"type": "noul", "instructions": "Is the customer angry?"}}}).encode()
    request = urllib.request.Request("http://127.0.0.1:8000/v1/systemone", data=body,
                                     headers={"content-type": "application/json"})
    for _ in range(120):  # wait for the model to load
        try:
            with urllib.request.urlopen(request) as reply:
                print(json.dumps(json.load(reply), indent=2))
            break
        except OSError:
            time.sleep(1)
    else:
        print("the server did not answer within 2 minutes")
finally:
    server.terminate()
    server.wait()

**Other options**

- `decisionformer predict CHECKPOINT requests.jsonl --output responses.jsonl` answers a file of requests.
- Merged Qwen3 code-readout checkpoints (from `deciformer15k_qwen3`) can be exported as a native
  `Qwen3ForSequenceClassification`: `decisionformer export CHECKPOINT exports/my-classifier --format sequence-classification`.
  See `docs/native-classifier.md`.
- Multi-GPU training: `torchrun --nproc-per-node 8 -m decisionformer train RECIPE`.
- QLoRA, multiple heads, prompt formats and attention modes are covered in the README.